In [8]:
import json
import re
import time
import numpy as np
import pandas as pd
from pathlib import Path

import yaml
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import timm
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, log_loss

In [9]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [10]:
CONFIG_PATH = "/content/drive/MyDrive/Deepfake_Preprocessed/config.yaml"

def load_config(path: str) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    print(f"config 로드 완료: {path}")
    return cfg

CFG = load_config(CONFIG_PATH)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

config 로드 완료: /content/drive/MyDrive/Deepfake_Preprocessed/config.yaml
Device: cuda


# Dataset

In [11]:
_image_size = CFG["input"]["image_size"]

train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((_image_size, _image_size)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.GaussianBlur(kernel_size=(5, 9), sigma=(0.1, 5)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((_image_size, _image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [12]:
class DeepfakeDataset(Dataset):
    def __init__(self, base_dir, df, transform=None, max_frames=64):
        self.base_dir = Path(base_dir)
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.max_frames = max_frames

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        frames_dir = self.base_dir / row["video_id"] / "frames"
        pngs = sorted(frames_dir.glob("*.png"))[: self.max_frames]

        if not pngs:
            raise RuntimeError(f"프레임 없음: {row['video_id']}")

        imgs = []
        for png in pngs:
            img = cv2.imread(str(png))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            imgs.append(img)

        # 부족한 프레임 마지막으로 패딩
        while len(imgs) < self.max_frames:
            imgs.append(imgs[-1].copy())

        if self.transform:
            seed = np.random.randint(0, 2**31)
            np.random.seed(seed)
            torch.manual_seed(seed)
            img_tensor = torch.stack([self.transform(img) for img in imgs])
        else:
            img_seq = np.stack(imgs, axis=0).transpose(0, 3, 1, 2)
            img_tensor = torch.from_numpy(img_seq).float() / 255.0

        label = torch.tensor(int(row["label"]), dtype=torch.long)
        source = str(row["source"])
        return img_tensor, label, source

# Model

In [13]:
class HighPassPreprocess(nn.Module):
    def __init__(self, strength=0.15):
        super().__init__()
        kernel = torch.tensor(
            [[0.0, -1.0, 0.0],
             [-1.0,  4.0, -1.0],
             [0.0, -1.0, 0.0]],
            dtype=torch.float32,
        )
        kernel = kernel.view(1, 1, 3, 3).repeat(3, 1, 1, 1)
        self.register_buffer("kernel", kernel)
        self.strength = strength

    def forward(self, x):
        hp = F.conv2d(x, self.kernel, padding=1, groups=3)
        return x + self.strength * hp

In [14]:
class ConvNeXtArtifactDetector(nn.Module):
    def __init__(self, model_name="convnext_tiny", pretrained=True, dropout=0.2, use_highpass=True):
        super().__init__()
        self.use_highpass = use_highpass
        self.pre = HighPassPreprocess(strength=0.15) if use_highpass else nn.Identity()

        self.backbone = timm.create_model(
            model_name,
            pretrained=pretrained,
            num_classes=0, # backbone을 feature 추출기로만 사용하기 [num_classes=0이면 timm이 classifier 안붙임]
            global_pool="avg"
        )

        feat_dim = self.backbone.num_features
        self.head = nn.Sequential( # Head 직접 만들어 붙임
            nn.LayerNorm(feat_dim),
            nn.Dropout(dropout),
            nn.Linear(feat_dim, 1)
        )

    def forward(self, x):
      # x : (B, T, C, H, W) | T: 영상에서 추출한 프레임 수
        B, T, C, H, W = x.shape
        x = x.view(B*T, C, H, W)
        x = self.pre(x)

        feat = self.backbone(x)
        feat = feat.view(B, T, -1).mean(dim=1)

        logit = self.head(feat).squeeze(1)
        return logit

# Scheduler

In [15]:
def build_scheduler(optimizer, cfg):
    """
    - warmup_epochs 동안 lr를 0 → lr로 선형 증가
    - 이후 Cosine Annealing으로 min_lr까지 감소
    """
    lr            = cfg["optimizer"]["lr"]
    warmup_epochs = cfg["scheduler"]["warmup_epochs"]
    min_lr        = cfg["scheduler"]["min_lr"]
    num_epochs    = cfg["training"]["num_epochs"]

    warmup = torch.optim.lr_scheduler.LinearLR(
        optimizer,
        start_factor=1e-6 / lr,
        end_factor=1.0,
        total_iters=warmup_epochs,
    )
    cosine = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=num_epochs - warmup_epochs,
        eta_min=min_lr,
    )
    return torch.optim.lr_scheduler.SequentialLR(
        optimizer,
        schedulers=[warmup, cosine],
        milestones=[warmup_epochs],
    )

In [16]:
@torch.no_grad()
def evaluate(model, loader):
    """
    Returns:
        auc   : ROC-AUC (높을수록 좋음)
        lloss : Log-loss (낮을수록 좋음, DFDC 공식 지표)
        acc   : Accuracy
    """
    model.eval()
    all_logits, all_labels = [], []

    for imgs, labels, sources in loader:
        imgs = imgs.to(device)
        logits = model(imgs)
        all_logits.append(logits.cpu())
        all_labels.append(labels)

    logits = torch.cat(all_logits)
    labels = torch.cat(all_labels).numpy()
    probs  = torch.sigmoid(logits).numpy()
    preds  = (probs >= 0.5).astype(int)

    auc   = roc_auc_score(labels, probs)
    lloss = log_loss(labels, probs)
    acc   = (preds == labels).mean()

    return {"auc": auc, "log_loss": lloss, "acc": acc}


def train_one_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    total_loss = 0.0

    for imgs, labels, sources in loader:
        imgs   = imgs.to(device)
        labels = labels.to(device).float()

        optimizer.zero_grad()


        # 훈련하다가 뜬 경고 메세지 없애기 !!  -> 이거랑 scaler도 함께 세트로 바꿔야 함.
        with torch.amp.autocast('cuda'):
        #with torch.cuda.amp.autocast():
            logits = model(imgs)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)


        # Gradient clipping — ConvNeXt + Transformer 계열에 권장
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    return total_loss / len(loader)

# CSV

In [17]:
import re
import json
import tarfile
import shutil
import pandas as pd
from pathlib import Path
from tqdm import tqdm

DRIVE_DIR     = Path("/content/drive/MyDrive/Deepfake_Preprocessed")
LOCAL_TMP_DIR = Path("/content/meta_tmp")
CSV_SAVE_PATH = DRIVE_DIR / "dataset_metadata.csv"

In [18]:
from concurrent.futures import ThreadPoolExecutor

if CSV_SAVE_PATH.exists():
    print(f"이미 존재함, 스킵: {CSV_SAVE_PATH}")
else:

    tar_files = sorted(DRIVE_DIR.glob("dataset_chunk_*.tar"))
    print(f"발견된 tar: {len(tar_files)}개 -> meta.json 병렬 추출 시작")

    def extract_meta_from_tar(tar_path):
        local_records=[]
        with tarfile.open(tar_path) as tar:
            meta_members = [m for m in tar.getmembers()
                           if m.name.endswith("meta.json")]
            for member in meta_members:
                f = tar.extractfile(member)
                if f:
                    data = json.load(f)
                    local_records.append(data)
        return local_records

    records = []
    with ThreadPoolExecutor(max_workers=8) as executor:
        results = list(tqdm(
            executor.map(extract_meta_from_tar, tar_files),
            total=len(tar_files),
            desc="병렬 추출 중"
        ))

    for r in results:
        records.extend(r)

    print(f"총 {len(records)}개 영상 메타데이터 수집 완료")

    # LOCAL_TMP_DIR.mkdir(parents=True, exist_ok=True)
    # tar_files = sorted(DRIVE_DIR.glob("dataset_chunk_*.tar"))
    # print(f"발견된 tar: {len(tar_files)}개 → meta.json 추출 시작")

    # records = []

    # for tar_path in tqdm(tar_files, desc="tar 순회 중"):
    #     with tarfile.open(tar_path) as tar:
    #         meta_members = [m for m in tar.getmembers() if m.name.endswith("meta.json")]
    #         for member in meta_members:
    #             tar.extract(member, path=LOCAL_TMP_DIR)

    # for meta_file in LOCAL_TMP_DIR.rglob("meta.json"):
    #     with open(meta_file, "r", encoding="utf-8") as f:
    #         records.append(json.load(f))

    # print(f"총 {len(records)}개 영상 메타데이터 수집 완료")

    def get_dfd_identity(path: Path):
        match = re.match(r"^(\d{2})(?:_\d{2})?__", path.stem)
        return int(match.group(1)) if match else None

    def get_split(video_path: str, source: str, meta_split:str= None) -> str:
        path = Path(video_path)

        if source == "CelebDF":
            return meta_split

        if source == "DFD":
            identity = get_dfd_identity(path)
            if identity is None:
                return "train"
            if identity <= 20:
                return "train"  # 1~22번 배우
            if identity <= 25:
                return "val"    # 23~25번 배우
            return "test"       # 26~28번 배우
        return "train" # CelebDF는 추후 추가

    def get_split_random(df: pd.DataFrame, seed: int=42) -> pd.DataFrame:
        from sklearn.model_selection import train_test_split

        train_idx, temp_idx = train_test_split(
            df.index, test_size=0.2, random_state=seed, stratify=df["label"]
        )
        val_idx, test_idx = train_test_split(
            temp_idx, test_size=0.5, random_state=seed, stratify=df.loc[temp_idx, "label"]
        )
        df["split"] = "train"
        df.loc[val_idx, "split"] = "val"
        df.loc[test_idx, "split"] = "test"
        return df

    df = pd.DataFrame(records)
    #df["split"] = df.apply(lambda row: get_split(row["video_path"], row["source"]), axis=1)
    #df = get_split_random(df)

    def assign_split(row):
        if row["source"] == "CelebDF":
            return row["split"]
        else:
            return get_split(row["video_path"], row["source"])
    df["split"] = df.apply(assign_split, axis=1)

    # ── 저장 ──
    df.to_csv(CSV_SAVE_PATH, index=False)
    print(f"\nCSV 저장 완료: {CSV_SAVE_PATH}")

    # 분할 결과 출력
    print("\n=== 분할 결과 ===")
    for split in ["train", "val", "test"]:
        sub = df[df["split"] == split]
        if sub.empty:
            continue
        print(f"  {split:5}: {len(sub):4}개 "
              f"(real={(sub.label==0).sum()}, fake={(sub.label==1).sum()})")

    shutil.rmtree(LOCAL_TMP_DIR)
    print("\n임시 폴더 정리 완료")

이미 존재함, 스킵: /content/drive/MyDrive/Deepfake_Preprocessed/dataset_metadata.csv


In [26]:
import time
import json
import tarfile
import shutil
import subprocess
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from torch.utils.data import DataLoader, WeightedRandomSampler
from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    confusion_matrix,
    balanced_accuracy_score,
)


def extract_single_tar(tar_path: Path, extract_root: str):
    tar_path = Path(tar_path)
    extract_root = Path(extract_root)

    if not tar_path.exists():
        raise FileNotFoundError(f"tar 파일이 없습니다: {tar_path}")

    extract_root.mkdir(parents=True, exist_ok=True)

    result = subprocess.run(
        ["tar", "-xf", str(tar_path), "-C", str(extract_root)],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )

    if result.returncode != 0:
        raise RuntimeError(
            f"tar 압축 해제 실패: {tar_path}\n"
            f"returncode: {result.returncode}\n"
            f"STDOUT:\n{result.stdout[-2000:]}\n"
            f"STDERR:\n{result.stderr[-4000:]}"
        )


def extract_chunks(tar_paths: list, extract_root: str = "/content") -> None:
    if len(tar_paths) == 0:
        raise ValueError("extract_chunks()에 tar 파일이 하나도 전달되지 않았습니다.")

    max_workers = min(4, len(tar_paths))

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [
            executor.submit(extract_single_tar, tar_path, extract_root)
            for tar_path in tar_paths
        ]

        for tar_path, future in zip(tar_paths, futures):
            try:
                future.result()
            except Exception as e:
                raise RuntimeError(f"tar 압축 해제 실패: {tar_path}") from e


def clear_local_crops(local_dir: Path) -> None:
    if local_dir.exists():
        shutil.rmtree(local_dir)
    local_dir.mkdir(parents=True, exist_ok=True)


def deduplicate_metadata(df: pd.DataFrame) -> pd.DataFrame:
    split_priority = {"test": 0, "val": 1, "train": 2}

    before = len(df)
    df = df.copy()
    df["_split_priority"] = df["split"].map(split_priority)

    df = (
        df.sort_values(["video_id", "_split_priority"])
          .drop_duplicates("video_id", keep="first")
          .drop(columns="_split_priority")
          .reset_index(drop=True)
    )

    print(f"metadata dedup: {before} -> {len(df)}")
    return df


def build_balanced_chunk_groups(all_tar_files, df, num_groups=5, cache_path=None):
    if cache_path is not None:
        cache_path = Path(cache_path)

    if cache_path is not None and cache_path.exists():
        tar_map = pd.read_csv(cache_path)
    else:
        records = []

        for tar_path in all_tar_files:
            with tarfile.open(tar_path) as tar:
                for member in tar.getmembers():
                    if not member.name.endswith("meta.json"):
                        continue

                    f = tar.extractfile(member)
                    if f is None:
                        continue

                    meta = json.load(f)
                    records.append({
                        "tar_name": tar_path.name,
                        "video_id": meta.get("video_id", Path(member.name).parent.name),
                    })

        tar_map = pd.DataFrame(records).drop_duplicates(["tar_name", "video_id"])

        if cache_path is not None:
            cache_path.parent.mkdir(parents=True, exist_ok=True)
            tar_map.to_csv(cache_path, index=False)

    # 같은 video_id가 여러 tar에 있으면 첫 번째 tar에만 배정
    tar_map = tar_map.drop_duplicates("video_id", keep="first")

    assigned = tar_map.merge(
        df[["video_id", "split", "source", "label"]],
        on="video_id",
        how="inner",
    )

    train_assigned = assigned[assigned["split"] == "train"]

    tar_stats = (
        train_assigned
        .groupby("tar_name")
        .agg(
            train_total=("video_id", "count"),
            train_real=("label", lambda s: int((s == 0).sum())),
            train_fake=("label", lambda s: int((s == 1).sum())),
        )
        .reset_index()
    )

    all_stats = pd.DataFrame({"tar_name": [p.name for p in all_tar_files]})
    tar_stats = all_stats.merge(tar_stats, on="tar_name", how="left").fillna(0)

    groups = [
        {"names": [], "real": 0, "fake": 0, "total": 0}
        for _ in range(num_groups)
    ]

    real_tars = tar_stats[tar_stats["train_real"] > 0].sort_values(
        "train_real", ascending=False
    )
    other_tars = tar_stats[tar_stats["train_real"] == 0].sort_values(
        "train_fake", ascending=False
    )

    def add_to_group(group, row):
        group["names"].append(row.tar_name)
        group["real"] += int(row.train_real)
        group["fake"] += int(row.train_fake)
        group["total"] += int(row.train_total)

    for row in real_tars.itertuples(index=False):
        k = min(range(num_groups), key=lambda i: (groups[i]["real"], groups[i]["total"]))
        add_to_group(groups[k], row)

    for row in other_tars.itertuples(index=False):
        k = min(range(num_groups), key=lambda i: groups[i]["total"])
        add_to_group(groups[k], row)

    tar_by_name = {p.name: p for p in all_tar_files}
    chunk_groups = [[tar_by_name[name] for name in g["names"]] for g in groups]

    group_video_ids = []
    print(f"균형 재구성된 tar group: {len(chunk_groups)}개")

    for i, g in enumerate(groups, 1):
        ids = set(assigned.loc[assigned["tar_name"].isin(g["names"]), "video_id"])
        group_video_ids.append(ids)

        group_rows = assigned[assigned["tar_name"].isin(g["names"])]
        print(
            f"\n[Balanced Group {i}] "
            f"tar={len(g['names'])}, train_real={g['real']}, "
            f"train_fake={g['fake']}, train_total={g['total']}"
        )
        print(group_rows.groupby(["split", "source", "label"]).size())

    return chunk_groups, group_video_ids


def train(cfg):
    p   = cfg["path"]
    m   = cfg["model"]
    inp = cfg["input"]
    tr  = cfg["training"]
    opt = cfg["optimizer"]
    ckp = cfg["checkpoint"]

    local_dir = Path(p["base_dir"])
    drive_dir = Path(p["save_dir"]).parent

    all_tar_files = sorted(drive_dir.glob("dataset_chunk_*.tar"))
    if len(all_tar_files) == 0:
        raise RuntimeError(f"tar 파일을 찾지 못했습니다: {drive_dir}")

    print(f"전체 tar: {len(all_tar_files)}개")

    #csv_path = drive_dir / "dataset_metadata.csv"
    #csv_path = drive_dir / "dataset_metadata_identity_split.csv"
    csv_path = drive_dir / "dataset_metadata_identity_stratified_split.csv"
    # 위 수정 경로는 데이터 누수 해결 이후의 경로임.


    raw_df = pd.read_csv(csv_path)
    df = deduplicate_metadata(raw_df)

    train_df = df[df["split"] == "train"].reset_index(drop=True)
    val_df   = df[df["split"] == "val"].reset_index(drop=True)

    print(f"\nTrain: {len(train_df)}개 | Val: {len(val_df)}개")
    print(f"  Train - real: {(train_df.label == 0).sum()}, fake: {(train_df.label == 1).sum()}")
    print(f"  Val   - real: {(val_df.label == 0).sum()}, fake: {(val_df.label == 1).sum()}")

    print("\n=== split / label ===")
    print(df.groupby(["split", "label"]).size())

    print(f"\n전체: {len(df)}개")
    print(f"real: {(df.label == 0).sum()}개")
    print(f"fake: {(df.label == 1).sum()}개")

    chunk_groups, group_video_ids = build_balanced_chunk_groups(
        all_tar_files,
        df,
        num_groups=5,
        cache_path=drive_dir / "tar_video_map.csv",
    )

    n_real = (train_df.label == 0).sum()
    n_fake = (train_df.label == 1).sum()

    pos_weight = torch.tensor([1.0], device=device)
    print(f"\npos_weight: {pos_weight.item():.3f}")
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    def get_sample_weights(df):
        count_map = {}
        for src in ["DFD", "CelebDF"]:
            for lbl in [0, 1]:
                count_map[(src, lbl)] = len(df[(df.source == src) & (df.label == lbl)])

        weights = []
        for _, row in df.iterrows():
            cnt = count_map.get((row["source"], row["label"]), 1)
            weights.append(1.0 / max(cnt, 1))

        return np.array(weights)

    global_sample_weights = get_sample_weights(train_df)

    train_df = train_df.copy()
    train_df["sample_weight"] = global_sample_weights

    print("\n[Sampler Weights per source/label]")
    for src in ["DFD", "CelebDF"]:
        for lbl in [0, 1]:
            mask = (train_df.source == src) & (train_df.label == lbl)
            if mask.sum() > 0:
                w = train_df.loc[mask, "sample_weight"].iloc[0]
                print(f"  {src} {'real' if lbl == 0 else 'fake'}: {w:.6f} (n={mask.sum()})")

    model = ConvNeXtArtifactDetector(
        model_name=m["model_name"],
        pretrained=True,
        dropout=m["dropout"],
        use_highpass=m["use_highpass"],
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=opt["lr"],
        weight_decay=opt["weight_decay"],
    )

    scheduler = build_scheduler(optimizer, cfg)
    scaler = torch.amp.GradScaler("cuda")

    #save_dir = Path(p["save_dir"])
    #save_dir.mkdir(parents=True, exist_ok=True)

    # 기존 save_dir의 parent가 데이터 루트로 쓰이므로,
    # checkpoints/하위 폴더가 아니라 Deepfake_Preprocessed 바로 아래 폴더로 둬야 함
    #CFG["path"]["save_dir"] = "/content/drive/MyDrive/Deepfake_Preprocessed/checkpoints_identity_split"
    CFG["path"]["save_dir"] = "/content/drive/MyDrive/Deepfake_Preprocessed/checkpoints_identity_stratified"

    save_dir = Path(CFG["path"]["save_dir"])
    save_dir.mkdir(parents=True, exist_ok=True)

    print("새 checkpoint 저장 위치:", save_dir)
    print("기존 last.pt 있음?", (save_dir / "last.pt").exists())

    best_checkpoints = []
    last_ckpt_path = save_dir / "last.pt"

    start_epoch = 1

    if last_ckpt_path.exists():
        latest_ckpt_path = last_ckpt_path
        print(f"기존 체크포인트가 있어, 이어서 학습하기: {latest_ckpt_path.name}")

        checkpoint = torch.load(latest_ckpt_path, map_location=device, weights_only=False)

        model.load_state_dict(checkpoint["model_state"])
        optimizer.load_state_dict(checkpoint["optimizer_state"])

        if "scheduler_state" in checkpoint:
            scheduler.load_state_dict(checkpoint["scheduler_state"])
        if "scaler_state" in checkpoint:
            scaler.load_state_dict(checkpoint["scaler_state"])

        start_epoch = checkpoint["epoch"] + 1
    else:
        print("기존 체크포인트 없음: 1번째 에포크부터 처음부터 학습을 시작합니다.")

    history = []
    print("\n학습 시작\n" + "=" * 60)

    for epoch in range(start_epoch, tr["num_epochs"] + 1):
        t0 = time.time()
        epoch_loss = 0.0
        n_chunks = 0

        # Train pass
        for group_idx, tar_group in enumerate(chunk_groups):
            print(
                f"\n  [Epoch {epoch} | Train Group {group_idx + 1}/{len(chunk_groups)}] "
                f"압축 해제 중: {[t.name for t in tar_group]}"
            )

            clear_local_crops(local_dir)
            extract_chunks(tar_group, extract_root=local_dir)

            extracted_ids = {path.parent.name for path in local_dir.glob("*/meta.json")}
            allowed_ids = group_video_ids[group_idx]
            available_ids = allowed_ids & extracted_ids

            missing_ids = allowed_ids - extracted_ids
            if missing_ids:
                print(f"  └ warning: 추출되지 않은 assigned video_id {len(missing_ids)}개")

            chunk_train_df = train_df[
                train_df["video_id"].isin(available_ids)
            ].reset_index(drop=True)

            if chunk_train_df.empty:
                print("  └ train 데이터 없음, 스킵")
                continue

            train_ds = DeepfakeDataset(
                local_dir,
                chunk_train_df,
                train_transform,
                inp["max_frame"],
            )

            sample_weights = chunk_train_df["sample_weight"].to_numpy()

            print(
                f"  └ sampler check: min={sample_weights.min():.6f}, "
                f"max={sample_weights.max():.6f}, n={len(sample_weights)}"
            )
            print(chunk_train_df.groupby(["source", "label"]).size())

            sampler = WeightedRandomSampler(
                weights=torch.tensor(sample_weights, dtype=torch.float),
                num_samples=len(sample_weights),
                replacement=True,
            )

            train_loader = DataLoader(
                train_ds,
                batch_size=tr["batch_size"],
                sampler=sampler,
                num_workers=tr["num_workers"],
                pin_memory=True,
            )

            chunk_loss = train_one_epoch(
                model,
                train_loader,
                optimizer,
                criterion,
                scaler,
            )

            epoch_loss += chunk_loss
            n_chunks += 1

            print(f"  └ train chunk_loss: {chunk_loss:.4f}")

        # Val pass
        epoch_val_loss = 0.0
        val_total_samples = 0
        all_val_logits, all_val_labels = [], []
        all_val_sources = []

        model.eval()

        for group_idx, tar_group in enumerate(chunk_groups):
            print(
                f"\n  [Epoch {epoch} | Val Group {group_idx + 1}/{len(chunk_groups)}] "
                f"압축 해제 중: {[t.name for t in tar_group]}"
            )

            clear_local_crops(local_dir)
            extract_chunks(tar_group, extract_root=local_dir)

            extracted_ids = {path.parent.name for path in local_dir.glob("*/meta.json")}
            allowed_ids = group_video_ids[group_idx]
            available_ids = allowed_ids & extracted_ids

            chunk_val_df = val_df[
                val_df["video_id"].isin(available_ids)
            ].reset_index(drop=True)

            if chunk_val_df.empty:
                print("  └ val 데이터 없음, 스킵")
                continue

            val_ds = DeepfakeDataset(
                local_dir,
                chunk_val_df,
                val_transform,
                inp["max_frame"],
            )

            val_loader = DataLoader(
                val_ds,
                batch_size=tr["batch_size"],
                shuffle=False,
                num_workers=tr["num_workers"],
                pin_memory=True,
            )

            with torch.no_grad():
                for imgs, labels, sources in val_loader:
                    imgs = imgs.to(device)
                    labels = labels.to(device)

                    logits = model(imgs)
                    loss = criterion(logits.view(-1), labels.float().view(-1))

                    batch_size_current = imgs.size(0)
                    epoch_val_loss += loss.item() * batch_size_current
                    val_total_samples += batch_size_current

                    all_val_logits.append(logits.cpu())
                    all_val_labels.append(labels.cpu())
                    all_val_sources.extend(list(sources))

            print(f"  └ val chunk 수집: {len(chunk_val_df)}개")
            print(chunk_val_df.groupby(["source", "label"]).size())

        model.train()

        avg_loss = epoch_loss / max(n_chunks, 1)
        avg_val_loss = epoch_val_loss / max(val_total_samples, 1)

        if len(all_val_logits) == 0:
            raise RuntimeError("이번 Epoch에서 Validation 데이터가 하나도 수집되지 않았습니다.")

        val_logits = torch.cat(all_val_logits)
        val_labels = torch.cat(all_val_labels).numpy()
        val_probs = torch.sigmoid(val_logits).numpy()
        val_preds = (val_probs >= 0.5).astype(int)

        cm = confusion_matrix(val_labels, val_preds)
        bal_acc = balanced_accuracy_score(val_labels, val_preds)

        print("\nconfusion_matrix:")
        print(cm)
        print(f"balanced_acc={bal_acc:.4f}")
        print(f"prob mean real={val_probs[val_labels == 0].mean():.4f}")
        print(f"prob mean fake={val_probs[val_labels == 1].mean():.4f}")
        print(f"prob min={val_probs.min():.4f}, max={val_probs.max():.4f}")

        val_metrics = {
            "auc": roc_auc_score(val_labels, val_probs),
            "f1": f1_score(val_labels, val_preds),
            "acc": (val_preds == val_labels).mean(),
            "balanced_acc": bal_acc,
            "loss": round(avg_val_loss, 4),
        }

        val_sources = np.array(all_val_sources)
        source_auc = {}

        for src in ["DFD", "CelebDF"]:
            mask = val_sources == src

            if mask.sum() == 0 or len(np.unique(val_labels[mask])) < 2:
                source_auc[src] = None
                print(f" [{src}] val_auc : 계산 불가")
                continue

            src_auc = roc_auc_score(val_labels[mask], val_probs[mask])
            source_auc[src] = src_auc

            print(
                f" [{src}] val_auc : {src_auc:.4f} "
                f"(real: {(val_labels[mask] == 0).sum()}, "
                f"fake: {(val_labels[mask] == 1).sum()})"
            )

        cur_lr = optimizer.param_groups[0]["lr"]
        scheduler.step()

        elapsed = time.time() - t0

        dfd_auc = source_auc["DFD"]
        celebdf_auc = source_auc["CelebDF"]
        selection_auc = dfd_auc if dfd_auc is not None else val_metrics["auc"]

        log = {
            "epoch": epoch,
            "train_loss": round(avg_loss, 4),
            "val_loss": round(avg_val_loss, 4),
            "val_auc": round(val_metrics["auc"], 4),
            "val_f1": round(val_metrics["f1"], 4),
            "val_acc": round(val_metrics["acc"], 4),
            "val_balanced_acc": round(val_metrics["balanced_acc"], 4),
            "dfd_auc": None if dfd_auc is None else round(dfd_auc, 4),
            "celebdf_auc": None if celebdf_auc is None else round(celebdf_auc, 4),
            "selection_auc": round(selection_auc, 4),
            "lr": round(cur_lr, 8),
        }

        history.append(log)

        print(
            f"\n[Epoch {epoch:02d}/{tr['num_epochs']}] "
            f"train_loss={log['train_loss']:.4f} | "
            f"val_loss={log['val_loss']:.4f} | "
            f"val_auc={log['val_auc']:.4f} | "
            f"val_bal_acc={log['val_balanced_acc']:.4f} | "
            f"val_f1={log['val_f1']:.4f} | "
            f"val_acc={log['val_acc']:.4f} | "
            f"lr={cur_lr:.2e} | "
            f"{elapsed:.1f}s"
        )

        checkpoint_payload = {
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "scaler_state": scaler.state_dict(),
            "val_metrics": val_metrics,
            "source_auc": source_auc,
            "selection_auc": selection_auc,
            "cfg": cfg,
        }

        torch.save(checkpoint_payload, last_ckpt_path)

        ckpt_path = save_dir / f"epoch{epoch:02d}_selauc{selection_auc:.4f}.pt"
        torch.save(checkpoint_payload, ckpt_path)

        best_checkpoints.append((selection_auc, ckpt_path))
        best_checkpoints.sort(key=lambda x: x[0], reverse=True)

        while len(best_checkpoints) > ckp["save_top_k"]:
            _, old_path = best_checkpoints.pop()
            if old_path.exists():
                old_path.unlink()
                print(f"  └ 삭제: {old_path.name}")

    print("\n" + "=" * 60)
    print("학습 완료!")

    if best_checkpoints:
        print(f"Best selection_auc: {best_checkpoints[0][0]:.4f} -> {best_checkpoints[0][1].name}")

    hist_df = pd.DataFrame(history)
    hist_path = save_dir / "train_history.csv"
    hist_df.to_csv(hist_path, index=False)
    print(f"학습 기록 저장: {hist_path}")

    clear_local_crops(local_dir)

    return model, history


if __name__ == "__main__":
    model, history = train(CFG)

전체 tar: 34개
metadata dedup: 9848 -> 9848

Train: 6325개 | Val: 1541개
  Train - real: 792, fake: 5533
  Val   - real: 215, fake: 1326

=== split / label ===
split  label
test   0         245
       1        1737
train  0         792
       1        5533
val    0         215
       1        1326
dtype: int64

전체: 9848개
real: 1252개
fake: 8596개
균형 재구성된 tar group: 5개

[Balanced Group 1] tar=7, train_real=192, train_fake=1099, train_total=1291
split  source   label
test   CelebDF  0         59
                1        146
       DFD      1        207
train  CelebDF  0        192
                1        395
       DFD      1        704
val    CelebDF  0         49
                1         59
       DFD      1        173
dtype: int64

[Balanced Group 2] tar=6, train_real=183, train_fake=981, train_total=1164
split  source   label
test   CelebDF  0         86
                1        206
       DFD      1        132
train  CelebDF  0        183
                1        606
       DFD      1   

KeyboardInterrupt: 

# Test

In [28]:
import torch
import numpy as np
import pandas as pd
from sklearn.metrics import roc_curve, roc_auc_score, f1_score, accuracy_score

def get_predictions(model, dataloader, device):
    """모델에 데이터를 넣고 확률(probs), 정답(labels), 출처(sources)를 뽑아내는 함수"""
    model.eval()
    all_probs, all_labels, all_sources = [], [], []

    with torch.no_grad():
        for imgs, labels, sources in dataloader:
            imgs = imgs.to(device)
            logits = model(imgs)
            probs = torch.sigmoid(logits).cpu().numpy()

            all_probs.extend(probs)
            all_labels.extend(labels.numpy())
            all_sources.extend(sources)

    return np.array(all_probs), np.array(all_labels), np.array(all_sources)


def run_final_evaluation(model, val_loader, test_loader, device):
    print("="*50)
    print(" [Step 1] Val 셋으로 최적의 임계값(Threshold) 탐색 중...")

    val_probs, val_labels, _ = get_predictions(model, val_loader, device)

    # Val 데이터 기준으로 Youden's J 최적화
    fpr, tpr, thresholds = roc_curve(val_labels, val_probs)
    optimal_idx = np.argmax(tpr - fpr)
    optimal_threshold = thresholds[optimal_idx]

    print(f" Val 데이터가 도출한 최적 임계값: {optimal_threshold:.4f}")
    print("="*50)


    print(" [Step 2] Test 셋으로 최종 성능 평가 시작...")
    test_probs, test_labels, test_sources = get_predictions(model, test_loader, device)

    # 💡 핵심: Test 데이터에는 스스로 임계값을 찾지 않고, Val에서 찾은 임계값을 '강제 적용'
    test_preds = (test_probs >= optimal_threshold).astype(int)

    # 전체 지표 계산
    total_auc = roc_auc_score(test_labels, test_probs)
    total_f1 = f1_score(test_labels, test_preds)
    total_acc = accuracy_score(test_labels, test_preds)

    print(f"\n [전체 Test 성적표]")
    print(f"AUC: {total_auc:.4f} | F1: {total_f1:.4f} | Acc: {total_acc:.4f}")
    print("-" * 50)

    # Source별 (DFD vs CelebDF) 세부 지표 계산
    for src in ["DFD", "CelebDF"]:
        mask = test_sources == src
        if mask.sum() == 0:
            continue

        src_labels = test_labels[mask]
        src_probs = test_probs[mask]
        src_preds = test_preds[mask]

        # Source별 평가 시, 한 클래스만 있을 경우의 Warning 방어
        if len(np.unique(src_labels)) < 2:
            print(f"[{src}] 데이터 불균형으로 평가 불가 (Real: {(src_labels==0).sum()}, Fake: {(src_labels==1).sum()})")
            continue

        src_auc = roc_auc_score(src_labels, src_probs)
        src_f1 = f1_score(src_labels, src_preds)
        src_acc = accuracy_score(src_labels, src_preds)

        print(f"[{src}] AUC: {src_auc:.4f} | F1: {src_f1:.4f} | Acc: {src_acc:.4f}")
        print(f"       (Real: {(src_labels==0).sum()}개, Fake: {(src_labels==1).sum()}개)")

    print("="*50)

# Test ver 2

In [30]:
# ===== Test용 모델 새로 생성 + checkpoint 로드 =====

import torch
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# best_ckpt_path는 앞 셀에서 찾았다고 가정
checkpoint = torch.load(best_ckpt_path, map_location=device, weights_only=False)

cfg = checkpoint.get("cfg", CFG)
model_cfg = cfg["model"]

test_model = ConvNeXtArtifactDetector(
    model_name=model_cfg.get("model_name", "convnext_tiny"),
    pretrained=False,
    dropout=model_cfg.get("dropout", 0.2),
    use_highpass=model_cfg.get("use_highpass", True),
).to(device)

state = checkpoint.get("model_state", checkpoint.get("model_state_dict"))
state = {k.replace("module.", ""): v for k, v in state.items()}

test_model.load_state_dict(state, strict=True)
test_model.eval()

print("loaded checkpoint:", best_ckpt_path)
print("epoch:", checkpoint.get("epoch"))
print("val_metrics:", checkpoint.get("val_metrics"))

loaded checkpoint: /content/drive/MyDrive/Deepfake_Preprocessed/checkpoints_identity_stratified/epoch17_selauc0.9142.pt
epoch: 17
val_metrics: {'auc': np.float64(0.9558613069556982), 'f1': 0.9498456790123457, 'acc': np.float64(0.9156391953277093), 'balanced_acc': np.float64(0.8827826300466519), 'loss': 0.633}


In [31]:
test_metrics = evaluate(test_model, test_loader)
print(test_metrics)

NameError: name 'test_loader' is not defined

In [32]:
from pathlib import Path
import re
import torch

CKPT_DIR = Path("/content/drive/MyDrive/Deepfake_Preprocessed/checkpoints_identity_stratified")

def get_score(path):
    ckpt = torch.load(path, map_location="cpu", weights_only=False)

    # 전체 val_auc 기준으로 best 선택
    if "val_metrics" in ckpt and "auc" in ckpt["val_metrics"]:
        return float(ckpt["val_metrics"]["auc"])

    # fallback: selection_auc 또는 파일명 selauc
    if "selection_auc" in ckpt:
        return float(ckpt["selection_auc"])

    m = re.search(r"selauc([0-9]+(?:\.[0-9]+)?)", path.name)
    return float(m.group(1)) if m else -1.0

ckpt_files = sorted(list(CKPT_DIR.glob("epoch*.pt")))

for p in ckpt_files:
    print(p.name, "score:", get_score(p))

best_ckpt_path = max(ckpt_files, key=get_score)
print("Best checkpoint:", best_ckpt_path)

checkpoint = torch.load(best_ckpt_path, map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state"])
model.eval()

epoch01_selauc0.5880.pt score: 0.6810375670840788
epoch02_selauc0.8566.pt score: 0.8674330913045003
epoch03_selauc0.8722.pt score: 0.9195026132098635
epoch06_selauc0.8610.pt score: 0.9300922515696798
epoch07_selauc0.8500.pt score: 0.888216001964292
epoch08_selauc0.8711.pt score: 0.9394068539759375
epoch09_selauc0.8976.pt score: 0.9397821740502998
epoch10_selauc0.8543.pt score: 0.925605948998562
epoch11_selauc0.8925.pt score: 0.9346890455645585
epoch15_selauc0.9023.pt score: 0.9298589918972956
epoch16_selauc0.9304.pt score: 0.9523097969062401
epoch17_selauc0.9142.pt score: 0.9558613069556982
Best checkpoint: /content/drive/MyDrive/Deepfake_Preprocessed/checkpoints_identity_stratified/epoch17_selauc0.9142.pt


NameError: name 'model' is not defined

# Test ver 3

In [33]:
import shutil
import subprocess
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score, confusion_matrix, balanced_accuracy_score, f1_score

drive_dir = Path("/content/drive/MyDrive/Deepfake_Preprocessed")
local_dir = Path(CFG["path"]["base_dir"])

csv_path = drive_dir / "dataset_metadata_identity_stratified_split.csv"
df = pd.read_csv(csv_path)

test_df = df[df["split"] == "test"].reset_index(drop=True)

print("Test:", len(test_df))
print(pd.crosstab(test_df["source"], test_df["label"]))

Test: 1982
label      0     1
source            
CelebDF  175  1141
DFD       70   596


In [34]:
all_tar_files = sorted(drive_dir.glob("dataset_chunk_*.tar"))
CHUNK_SIZE = 8
chunk_groups = [
    all_tar_files[i:i+CHUNK_SIZE]
    for i in range(0, len(all_tar_files), CHUNK_SIZE)
]

all_probs = []
all_labels = []
all_sources = []

test_model.eval()

for gi, group in enumerate(chunk_groups, 1):
    print(f"\n[Test Group {gi}/{len(chunk_groups)}] 압축 해제 중:", [p.name for p in group])

    clear_local_crops(local_dir)
    extract_chunks(group, extract_root="/content")

    # 이번 group에 실제로 압축 해제된 video_id만 평가
    available_ids = {p.name for p in local_dir.iterdir() if p.is_dir()}
    group_test_df = test_df[test_df["video_id"].isin(available_ids)].reset_index(drop=True)

    print("  └ test chunk 수집:", len(group_test_df))
    if len(group_test_df) == 0:
        continue

    print(group_test_df.groupby(["source", "label"]).size())

    test_dataset = DeepfakeDataset(
        local_dir,
        group_test_df,
        transform=val_transform,
        max_frames=CFG["input"]["max_frame"],
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=CFG["training"]["batch_size"],
        shuffle=False,
        num_workers=CFG["training"].get("num_workers", 2),
        pin_memory=True,
    )

    with torch.no_grad():
        for imgs, labels, sources in test_loader:
            imgs = imgs.to(device)
            logits = test_model(imgs)
            probs = torch.sigmoid(logits).detach().cpu().numpy()

            all_probs.extend(probs.tolist())
            all_labels.extend(labels.numpy().tolist())
            all_sources.extend(list(sources))

clear_local_crops(local_dir)

all_probs = np.array(all_probs)
all_labels = np.array(all_labels)
all_sources = np.array(all_sources)

preds = (all_probs >= 0.5).astype(int)


[Test Group 1/5] 압축 해제 중: ['dataset_chunk_000.tar', 'dataset_chunk_001.tar', 'dataset_chunk_002.tar', 'dataset_chunk_003.tar', 'dataset_chunk_004.tar', 'dataset_chunk_005.tar', 'dataset_chunk_006.tar', 'dataset_chunk_007.tar']
  └ test chunk 수집: 0

[Test Group 2/5] 압축 해제 중: ['dataset_chunk_008.tar', 'dataset_chunk_009.tar', 'dataset_chunk_010.tar', 'dataset_chunk_011.tar', 'dataset_chunk_012.tar', 'dataset_chunk_013.tar', 'dataset_chunk_014.tar', 'dataset_chunk_015.tar']
  └ test chunk 수집: 0

[Test Group 3/5] 압축 해제 중: ['dataset_chunk_016.tar', 'dataset_chunk_017.tar', 'dataset_chunk_018.tar', 'dataset_chunk_019.tar', 'dataset_chunk_020.tar', 'dataset_chunk_021.tar', 'dataset_chunk_022.tar', 'dataset_chunk_023.tar']
  └ test chunk 수집: 0

[Test Group 4/5] 압축 해제 중: ['dataset_chunk_024.tar', 'dataset_chunk_025.tar', 'dataset_chunk_026.tar', 'dataset_chunk_027.tar', 'dataset_chunk_028.tar', 'dataset_chunk_029.tar', 'dataset_chunk_030.tar', 'dataset_chunk_031.tar']
  └ test chunk 수집: 0

[Te

In [35]:
print("Test samples:", len(all_labels))
print("confusion_matrix:")
print(confusion_matrix(all_labels, preds))

print("balanced_acc:", balanced_accuracy_score(all_labels, preds))
print("f1:", f1_score(all_labels, preds))
print("acc:", (preds == all_labels).mean())
print("auc:", roc_auc_score(all_labels, all_probs))

for src in ["DFD", "CelebDF"]:
    mask = all_sources == src

    if mask.sum() == 0 or len(np.unique(all_labels[mask])) < 2:
        print(f"[{src}] test_auc: 계산 불가")
        continue

    print(
        f"[{src}] test_auc: {roc_auc_score(all_labels[mask], all_probs[mask]):.4f} "
        f"(real: {(all_labels[mask] == 0).sum()}, fake: {(all_labels[mask] == 1).sum()})"
    )

Test samples: 0
confusion_matrix:
[]
balanced_acc: nan
f1: 0.0
acc: nan


/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/tmp/ipykernel_10718/519239042.py:7: RuntimeWarning: Mean of empty slice.
  print("acc:", (preds == all_labels).mean())
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


ValueError: Found array with 0 sample(s) (shape=(0,)) while a minimum of 1 is required.

In [36]:
print("local_dir:", local_dir)
print("exists:", local_dir.exists())

print("\nchildren sample:")
for p in list(local_dir.iterdir())[:30]:
    print(p, "dir=", p.is_dir())

print("\nframes dir sample:")
for p in list(local_dir.rglob("frames"))[:20]:
    print(p)

print("\npng sample:")
for p in list(local_dir.rglob("*.png"))[:20]:
    print(p)

print("\ntest video_id sample:")
print(test_df["video_id"].head(10).tolist())

local_dir: /content/processed_crops
exists: True

children sample:

frames dir sample:

png sample:

test video_id sample:
['celebdf_00001_real', 'celebdf_00009_real', 'celebdf_00014_real', 'celebdf_00020_real', 'celebdf_00026_real', 'celebdf_00032_real', 'celebdf_00037_real', 'celebdf_00040_real', 'celebdf_00044_real', 'celebdf_00051_real']


# <기타: 데이터 누수 확인>

In [ ]:
# import pandas as pd
# from itertools import combinations

# #meta = pd.read_csv("/content/drive/MyDrive/.../dataset_metadata.csv")
# meta = pd.read_csv("/content/drive/MyDrive/Deepfake_Preprocessed/dataset_metadata.csv")

# print(meta.shape)
# print(meta["split"].value_counts())
# print(pd.crosstab(meta["split"], meta["label"], margins=True))

# # 1. 같은 video_id가 여러 split에 있는지
# split_sets = {
#     s: set(meta.loc[meta["split"] == s, "video_id"])
#     for s in ["train", "val", "test"]
# }

# for a, b in combinations(["train", "val", "test"], 2):
#     overlap = split_sets[a] & split_sets[b]
#     print(f"{a}-{b} video_id overlap:", len(overlap))
#     print(list(sorted(overlap))[:20])

# # 2. video_id별 split 개수 확인
# leaky_video_ids = (
#     meta.groupby("video_id")["split"]
#     .nunique()
#     .loc[lambda x: x > 1]
# )

# print("video_id split conflict:", len(leaky_video_ids))
# display(meta[meta["video_id"].isin(leaky_video_ids.index)]
#         .sort_values(["video_id", "split"])
#         [["video_id", "label", "source", "video_path", "split"]]
#         .head(100))

In [ ]:
# import re

# def extract_identity(row):
#     vid = row["video_id"]

#     # CelebDF: celebdf_id39_0003_real / celebdf_id39_id40_0001 ...
#     m = re.search(r"celebdf_(id\d+)", vid)
#     if m:
#         return f"CelebDF_{m.group(1)}"

#     # DFD는 video_id 규칙에 맞게 더 정교하게 조정 필요
#     # 일단 마지막 token이나 앞쪽 hash를 후보로 볼 수 있음
#     if vid.startswith("dfd_"):
#         parts = vid.split("_")
#         return "DFD_" + parts[1] if len(parts) > 1 else vid

#     return vid

# meta["identity_guess"] = meta.apply(extract_identity, axis=1)

# identity_splits = meta.groupby("identity_guess")["split"].nunique()
# leaky_identities = identity_splits[identity_splits > 1].index

# print("identity overlap:", len(leaky_identities))
# display(meta[meta["identity_guess"].isin(leaky_identities)]
#         .sort_values(["identity_guess", "split"])
#         [["identity_guess", "video_id", "label", "source", "split"]]
#         .head(100))

In [ ]:
# dedup_meta = deduplicate_metadata(meta)

# from itertools import combinations

# split_sets = {
#     s: set(dedup_meta.loc[dedup_meta["split"] == s, "video_id"])
#     for s in ["train", "val", "test"]
# }

# for a, b in combinations(["train", "val", "test"], 2):
#     overlap = split_sets[a] & split_sets[b]
#     print(f"[dedup] {a}-{b} video_id overlap:", len(overlap))
#     print(list(sorted(overlap))[:20])

# print(dedup_meta.shape)
# print(dedup_meta["split"].value_counts())
# print(pd.crosstab(dedup_meta["split"], dedup_meta["label"], margins=True))

In [ ]:
# leaky_mask = dedup_meta["celeb_id"].isin(leaky_ids)

# print("전체 leaky rows:", leaky_mask.sum())
# print(pd.crosstab(dedup_meta.loc[leaky_mask, "split"],
#                   dedup_meta.loc[leaky_mask, "label"],
#                   margins=True))

# print("val 중 leaky identity 비율:")
# val = dedup_meta[dedup_meta["split"] == "val"]
# print(val["celeb_id"].isin(leaky_ids).mean())

# print("test 중 leaky identity 비율:")
# test = dedup_meta[dedup_meta["split"] == "test"]
# print(test["celeb_id"].isin(leaky_ids).mean())

In [ ]:
# train_ids = set(
#     dedup_meta.loc[dedup_meta["split"] == "train", "celeb_id"].dropna()
# )

# val_nonleaky = dedup_meta[
#     (dedup_meta["split"] == "val") &
#     (~dedup_meta["celeb_id"].isin(train_ids))
# ]

# print(len(val_nonleaky))
# print(pd.crosstab(val_nonleaky["source"], val_nonleaky["label"], margins=True))

In [ ]:
# import re
# import pandas as pd
# from pathlib import Path
# from itertools import combinations
# from sklearn.model_selection import StratifiedGroupKFold

# drive_dir = Path("/content/drive/MyDrive/Deepfake_Preprocessed")

# meta = pd.read_csv(drive_dir / "dataset_metadata.csv")
# meta = deduplicate_metadata(meta)

# def extract_group(row):
#     vid = row["video_id"]

#     if row["source"] == "CelebDF":
#         ids = re.findall(r"id\d+", vid)
#         if ids:
#             return "CelebDF_" + "_".join(sorted(set(ids)))

#     if row["source"] == "DFD":
#         return "DFD_" + vid

#     return row["source"] + "_" + vid

# meta["group_id"] = meta.apply(extract_group, axis=1)
# meta["stratum"] = meta["source"].astype(str) + "_" + meta["label"].astype(str)

# # 1차: trainval vs test
# sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
# trainval_idx, test_idx = next(
#     sgkf.split(meta, y=meta["stratum"], groups=meta["group_id"])
# )

# trainval_df = meta.iloc[trainval_idx].copy()
# test_df = meta.iloc[test_idx].copy()

# # 2차: train vs val
# sgkf2 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=43)
# train_idx, val_idx = next(
#     sgkf2.split(
#         trainval_df,
#         y=trainval_df["stratum"],
#         groups=trainval_df["group_id"],
#     )
# )

# train_df = trainval_df.iloc[train_idx].copy()
# val_df = trainval_df.iloc[val_idx].copy()

# train_df["split"] = "train"
# val_df["split"] = "val"
# test_df["split"] = "test"

# new_meta = pd.concat([train_df, val_df, test_df], ignore_index=True)

# for a, b in combinations(["train", "val", "test"], 2):
#     ga = set(new_meta.loc[new_meta["split"] == a, "group_id"])
#     gb = set(new_meta.loc[new_meta["split"] == b, "group_id"])
#     print(a, b, "group overlap:", len(ga & gb))

# print(new_meta["split"].value_counts())
# print(pd.crosstab(new_meta["split"], new_meta["label"], margins=True))
# print(pd.crosstab([new_meta["split"], new_meta["source"]], new_meta["label"]))

# save_path = drive_dir / "dataset_metadata_identity_stratified_split.csv"
# new_meta.to_csv(save_path, index=False)
# print("saved:", save_path)